# Modelling Random Forest - 4-Split (train_core + val + train_full + test)

**Project:** Prediksi PM2.5 di Jakarta - Eksperimen Tambahan

**Tujuan notebook ini:** uji apakah strategi split 4-bagian (train_core + validation + train_full + test) memberi hasil berbeda dibanding 3-split (train + val + test) yang sudah dipakai di notebook utama. Dilakukan pada **dataset v1 (proposal scope) dan v2 (out-of-scope, dataset bersih)**.

**Date boundaries (sesuai setup awal user):**
- Train_core : 2022-01-01 -> 2024-01-15
- Validation : 2024-01-16 -> 2024-05-26
- Train_full : 2022-01-01 -> 2024-05-26 (train_core + val)
- Test       : 2024-05-27 -> 2025-01-01

**Workflow:**
1. Tune hyperparameter pakai **train_core**, evaluasi di **val**
2. Pilih best by **val_RMSE**
3. Retrain final pada **train_full** (train_core + val)
4. Evaluasi sekali pada **test**

**Note:** notebook ini self-contained. Untuk reproducibility, kode utama juga tersedia di `src/eksperimen_4split.py`.

## 1. Setup & Functions

In [1]:
import os, time, warnings, json
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM = 42
np.random.seed(RANDOM)
DIR_OUT = Path('outputs')
TARGET = 'ISPU PM2.5'

# Boundaries 4-split
BATAS_VAL_MULAI  = pd.Timestamp('2024-01-16')
BATAS_VAL_AKHIR  = pd.Timestamp('2024-05-26')
BATAS_TEST_MULAI = pd.Timestamp('2024-05-27')
print('Setup OK')

Setup OK


In [2]:
def load_dan_fe(versi):
    """versi = 'v1' atau 'v2'. Returns df + fitur info."""
    if versi == 'v1':
        path = Path('../data/final for modelling/dataset_final_model.csv')
        kolom_cuaca = ['temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip']
    else:
        path = Path('../data/final for modelling/dataset_final_model_v2.csv')
        kolom_cuaca = ['temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip',
                       'cloudcover', 'tempmax', 'tempmin', 'feelslike', 'uvindex',
                       'precipprob', 'sealevelpressure', 'dew', 'winddir_sin', 'winddir_cos']
    df = pd.read_csv(path)
    df['tanggal'] = pd.to_datetime(df['tanggal'])
    kolom_stasiun = [c for c in df.columns if c.startswith('station_')]
    df['station'] = df[kolom_stasiun].idxmax(axis=1).str.replace('station_', '', regex=False)
    if 'bulan' not in df.columns: df['bulan'] = df['tanggal'].dt.month
    if 'hari_minggu' not in df.columns: df['hari_minggu'] = df['tanggal'].dt.dayofweek
    df['bulan_sin'] = np.sin(2*np.pi*df['bulan']/12); df['bulan_cos'] = np.cos(2*np.pi*df['bulan']/12)
    df['hari_minggu_sin'] = np.sin(2*np.pi*df['hari_minggu']/7); df['hari_minggu_cos'] = np.cos(2*np.pi*df['hari_minggu']/7)
    def musim(b):
        if b in (11,12,1,2,3): return 'Hujan'
        if b == 4: return 'Transisi'
        return 'Kemarau'
    df['musim'] = df['bulan'].map(musim)
    for lag in [1,3,7]: df[f'pm25_lag_{lag}'] = df.groupby('station')[TARGET].shift(lag)
    prev = df.groupby('station')[TARGET].shift(1)
    def rs(s,w,f): return s.groupby(df['station']).rolling(w).agg(f).reset_index(level=0, drop=True)
    df['pm25_rolling_mean_3'] = rs(prev,3,'mean'); df['pm25_rolling_mean_7'] = rs(prev,7,'mean')
    df['pm25_rolling_max_7']  = rs(prev,7,'max');  df['pm25_rolling_std_7']  = rs(prev,7,'std')
    fitur_lag = ['pm25_lag_1','pm25_lag_3','pm25_lag_7','pm25_rolling_mean_3',
                 'pm25_rolling_mean_7','pm25_rolling_max_7','pm25_rolling_std_7']
    kolom_cuaca = [c for c in kolom_cuaca if c in df.columns]
    df = df.dropna(subset=fitur_lag + [TARGET]).reset_index(drop=True)
    return df, kolom_cuaca, fitur_lag, kolom_stasiun

def split_4(df):
    return (df['tanggal'] < BATAS_VAL_MULAI,
            (df['tanggal'] >= BATAS_VAL_MULAI) & (df['tanggal'] <= BATAS_VAL_AKHIR),
            df['tanggal'] <= BATAS_VAL_AKHIR,
            df['tanggal'] >= BATAS_TEST_MULAI)

def ev(y, yp):
    return {'MAE': float(mean_absolute_error(y,yp)),
            'RMSE': float(np.sqrt(mean_squared_error(y,yp))),
            'R2': float(r2_score(y,yp))}

print('Functions ready')

Functions ready


In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import PredefinedSplit, GridSearchCV

def run_rf_4split(df, kolom_cuaca, fitur_lag, kolom_stasiun, label):
    NUM = kolom_cuaca + fitur_lag + ['bulan_sin','bulan_cos','hari_minggu_sin','hari_minggu_cos'] + kolom_stasiun
    FITUR = NUM + ['musim']
    m_core, m_val, m_full, m_test = split_4(df)
    X_core, X_val, X_full, X_test = df.loc[m_core, FITUR], df.loc[m_val, FITUR], df.loc[m_full, FITUR], df.loc[m_test, FITUR]
    y_core, y_val, y_full, y_test = df.loc[m_core, TARGET], df.loc[m_val, TARGET], df.loc[m_full, TARGET], df.loc[m_test, TARGET]
    print(f'  Train_core {len(X_core)} | Val {len(X_val)} | Train_full {len(X_full)} | Test {len(X_test)}')

    def buat_pipe(params=None):
        p = dict(random_state=RANDOM, n_jobs=-1)
        if params: p.update(params)
        pre = ColumnTransformer([
            ('num', SimpleImputer(strategy='median'), NUM),
            ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                              ('oh',OneHotEncoder(handle_unknown='ignore'))]), ['musim']),
        ])
        return Pipeline([('pre',pre), ('model', RandomForestRegressor(**p))])

    # Tune Grid Search di train_core + val (via PredefinedSplit)
    pen = np.r_[np.full(len(X_core),-1), np.zeros(len(X_val), dtype=int)]
    Xg = pd.concat([X_core, X_val]).reset_index(drop=True)
    yg = pd.concat([y_core, y_val]).reset_index(drop=True)
    grid = {
        'model__n_estimators': [400, 600],
        'model__max_depth': [None, 15, 20],
        'model__min_samples_leaf': [1, 2],
        'model__min_samples_split': [2, 5],
    }
    gs = GridSearchCV(buat_pipe(), grid, cv=PredefinedSplit(test_fold=pen),
                      scoring='neg_root_mean_squared_error', n_jobs=-1, refit=False, verbose=1)
    gs.fit(Xg, yg)
    params_best = {k.replace('model__',''): v for k,v in gs.best_params_.items()}
    val_rmse = -gs.best_score_
    print(f'  Best params: {params_best}')
    print(f'  Best val_RMSE: {val_rmse:.4f}')

    # Retrain pada train_full lalu eval di test
    model_final = buat_pipe(params_best)
    model_final.fit(X_full, y_full)
    metric_test = ev(y_test, model_final.predict(X_test))
    print(f'  Test: RMSE {metric_test["RMSE"]:.4f} | R2 {metric_test["R2"]:.4f} | MAE {metric_test["MAE"]:.4f}')
    return {'model':'Random Forest','dataset':label,'tuning':'Grid Search',
            'val_RMSE': round(val_rmse,4),'test_RMSE': round(metric_test['RMSE'],4),
            'test_R2': round(metric_test['R2'],4),'test_MAE': round(metric_test['MAE'],4),
            'params': params_best}

print('RF function ready')

RF function ready


## 2. RF pada Dataset v1 (proposal scope, 6 cuaca)

In [4]:
df_v1, cuaca_v1, fitur_lag, kolom_stasiun = load_dan_fe('v1')
print(f'v1: shape {df_v1.shape}, {len(cuaca_v1)} cuaca')
hasil_v1 = run_rf_4split(df_v1, cuaca_v1, fitur_lag, kolom_stasiun, 'v1 (6 cuaca)')

v1: shape (7630, 31), 6 cuaca
  Train_core 5166 | Val 924 | Train_full 6090 | Test 1540
Fitting 1 folds for each of 24 candidates, totalling 24 fits
  Best params: {'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 600}
  Best val_RMSE: 11.5117
  Test: RMSE 12.3091 | R2 0.6461 | MAE 8.9574


## 3. RF pada Dataset v2 (OOS, 15 cuaca)

In [5]:
df_v2, cuaca_v2, fitur_lag, kolom_stasiun = load_dan_fe('v2')
print(f'v2: shape {df_v2.shape}, {len(cuaca_v2)} cuaca')
hasil_v2 = run_rf_4split(df_v2, cuaca_v2, fitur_lag, kolom_stasiun, 'v2 (15 cuaca)')

v2: shape (5560, 43), 15 cuaca
  Train_core 3679 | Val 727 | Train_full 4406 | Test 1154
Fitting 1 folds for each of 24 candidates, totalling 24 fits
  Best params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 400}
  Best val_RMSE: 13.4955
  Test: RMSE 14.4454 | R2 0.5843 | MAE 10.5524


## 4. Perbandingan v1 vs v2

In [6]:
df_perbandingan = pd.DataFrame([hasil_v1, hasil_v2])
df_perbandingan.to_csv(DIR_OUT/'rf_4split_v1_v2.csv', index=False)
display(df_perbandingan[['model','dataset','val_RMSE','test_RMSE','test_R2','test_MAE']])

,model,dataset,val_RMSE,test_RMSE,test_R2,test_MAE
0,Random Forest,v1 (6 cuaca),11.5117,12.3091,0.6461,8.9574
1,Random Forest,v2 (15 cuaca),13.4955,14.4454,0.5843,10.5524


## 5. Interpretasi

- **v1**: dataset proposal, 6 fitur cuaca, ~6090 train_full
- **v2**: dataset bersih (drop 24% imputasi), 15-16 fitur cuaca, ~4400 train_full

Bandingkan val_RMSE dan test_R^2:
- val_RMSE turun di v2? Mungkin overfit ke val (val periodenya beda).
- test_R^2 v1 vs v2? v1 cenderung lebih tinggi karena test set v1 ada baris imputasi yang mudah ditebak.

**File:** `rf_4split_v1_v2.csv`. Semua eksperimen 4-split lintas 3 model + 2 dataset ada di `perbandingan_4split_3model_2dataset.csv`.